### Instalacja biliotek

In [1]:
#pip install -r requirements.txt

### Model

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPEN_AI_KEY")

client = OpenAI(
    api_key=api_key,
)

In [3]:
import ipywidgets as widgets
from utils.AudioManager import VoiceRecorder

output_area = widgets.Output()
display(output_area)

MODEL_NAME = "whisper-1" 
wavPath = r"C:\Users\szymo\Desktop\studia\mall-center-ai-assistant\voice\input.wav"
textPath = r"C:\Users\szymo\Desktop\studia\mall-center-ai-assistant\voice\transcription.txt"

# Inicjalizacja VoiceRecorder
recorder = VoiceRecorder(output_area=output_area,openai_client=client,wav_path=wavPath, txt_path=textPath)

btn_start = widgets.Button(description="Start", icon="microphone")
btn_stop = widgets.Button(description="Stop", icon="stop")
btn_transcribe = widgets.Button(description="Transkrybuj", icon="file-text")

btn_start.on_click(lambda x: recorder.start_recording())
btn_stop.on_click(lambda x: recorder.stop_recording())
btn_transcribe.on_click(lambda x: recorder.transcribe())

display(widgets.HBox([btn_start, btn_stop, btn_transcribe]))

Output()

### Połączenie kategorii z danym sklepem

In [4]:
import pandas as pd
from utils.ShopAssistant import ShopAssistant

df = pd.read_csv("./db/Posnania_db(Shops).csv", sep=";")
df.head()

,ID,NAME,CATEGORIES
0,1,4F,"Odzież damska, Odzież męska, Odzież dziecięca,..."
1,2,A.Blikle,"Ciasta, wypieki, Cukierki, słodycze, Torty, de..."
2,3,Adidas,"Odzież damska, Odzież męska, Odzież dziecięca,..."
3,4,Adopt,"Perfumy, Kosmetyki (makijaż), Pielęgnacja ciał..."
4,5,Apart,"Biżuteria damska, Biżuteria męska, Zegarki, Ob..."


In [5]:
assistant = ShopAssistant(df, client)

with open("./voice/transcription.txt", "r", encoding="utf-8") as f:
    text = f.read()

result = assistant.analyze_intent(question=text)
print("Analiza intencji:", result)

Asystent załadowany. Znaleziono 708 unikalnych kategorii.
Analiza intencji: {'input_text': 'Chciałbym kupić młotek.', 'detected_categories': ['Narzędzia'], 'matching_shops': [{'name': 'Leroy Merlin', 'id': 104}]}


### Pobranie odpowiednich produktow

In [6]:
df_products = pd.read_csv("./db/Posnania_db(Products).csv", sep=";")
df_products.head()

target_shops = result['matching_shops']
shop_ids = [shop['id'] for shop in target_shops]
shop_inventory = df_products[df_products['SHOP_ID'].isin(shop_ids)]
print(f"--- Asortyment w sklepach: {target_shops} ---")
display(shop_inventory)

--- Asortyment w sklepach: [{'name': 'Leroy Merlin', 'id': 104}] ---


,ID,SHOP_ID,NAME,SECTION
415,416,104,Wiertarka elektryczna,Narzędzia
416,417,104,Zestaw młotków i kluczy,Narzędzia
417,418,104,Płyty OSB 18mm,Materiały remontowe
418,419,104,Lampka stołowa LED,Wyposażenie domu i ogrodu
419,420,104,Doniczka ceramiczna,Wyposażenie domu i ogrodu


### Przekazanie produktow modelowi

In [7]:
system_prompt = """
Jesteś asystentem analizującym bazę danych produktów dostępnych w sklepie.
użytkownik zapyta Cię o dostępność określonych produktów lub powie ci cel wizyty w naszej galerii handlowej.
otrzymasz listę sklepów, które mogą spełniać jego potrzeby.
Twoim zadaniem jest wygenerowanie przyjaznej odpowiedzi, która podsumowuje te informacje i zachęca użytkownika do odwiedzenia tych sklepów.
Pamiętaj, aby odpowiedź była zwięzła i uprzejma.
Jeżeli nie ma pasujących sklepów, uprzejmie poinformuj użytkownika, że nie znaleziono odpowiednich opcji.
Nie dokładaj żadnych dodatkowych informacji ani nie sugeruj innych sklepów, cen czy godzin otwarcia bo na to danych nie posiadamy.
Nie pytaj czy pomoc jeszcze w czymś.
"""

In [8]:
products_text = shop_inventory.to_string(index=False)
query = (
    text + "\n\n"
    "Na podstawie poniższej listy sklepów oraz ich produktów, wygeneruj przyjazną odpowiedź dla użytkownika:\n\n"
    f"Sklepy:\n{', '.join([shop['name'] for shop in target_shops])}\n\n"
    f"Produkty:\n{products_text}"
)
messages = [
    {"role": "developer", "content": system_prompt},        
    {"role": "user", "content": query}
]

In [9]:
response = client.responses.create(
    model="gpt-5-mini",
    input=messages
)

In [10]:
print(response.output_text)

Świetnie — w Leroy Merlin znalazłem produkt pasujący do Twojego zapytania: „Zestaw młotków i kluczy” (sekcja: Narzędzia). Zachęcam do odwiedzenia Leroy Merlin, gdzie możesz go znaleźć.


### Odpowiedz głosowa

In [11]:
from IPython.display import Audio
audio_response = client.audio.speech.create(
    model="tts-1",  
    voice="alloy",
    input=response.output_text,
    speed=1.2
)
audio_bytes = audio_response.read()
Audio(audio_bytes, autoplay=True)